In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import dabench as dab
import numpy as np
import jax
from timeit import default_timer as timer
import pandas as pd
import pickle
import os

# Set up

### Read-in Raytune Results

In [3]:
raytune_system_dim_results = pd.read_csv('./out_rev/l96/raytune_l96_v2.csv')
raytune_system_dim_results['trialnum'] = raytune_system_dim_results.index
raytune_system_dim_results.index = np.arange(raytune_system_dim_results.shape[0])
rows_to_get = raytune_system_dim_results.groupby(['system_dim']).idxmin(numeric_only=True)['rmse']
best_results_system_dim = raytune_system_dim_results.loc[rows_to_get]

In [4]:
raytune_heatmap_results = pd.read_csv('./out_rev/l96/raytune_l96_heatmap_v2.csv')
raytune_heatmap_results['trialnum'] = raytune_heatmap_results.index
raytune_heatmap_results.index = np.arange(raytune_heatmap_results.shape[0])
rows_to_get = raytune_heatmap_results.groupby(['num_obs','obs_sd']).idxmin(numeric_only=True)['rmse']
best_results_heatmap = raytune_heatmap_results.loc[rows_to_get]

### Define parameters

In [5]:
system_dim= 36
spinup_size = 14400
valid_size = 5000
transient_size = 1000
test_size = 5000
nr_steps = spinup_size + valid_size + transient_size + test_size
delta_t=0.01
obs_sd = 0.5
sigma_obs = obs_sd *1.25
analysis_window = 0.1
analysis_time_in_window = 0.0
obs_location_count = 18
num_iters = 3
n_outer_loops = 3
num_runs = 1
output_suffix='v2'

### Function definition: Backprop 4DVar

We'll need to prep and run Backprop-4DVar many times, so this wraps it all into one function

In [6]:
def calc_background_cov(enkf_run_pkl):
    with open(enkf_run_pkl, 'rb') as f: 
        etkf_run = pickle.load(f) 
    system_dim = etkf_run.values.shape[-1]
    etkf_errors = np.moveaxis(etkf_run.values, (0,1,2), (1,0,2)) - np.mean(etkf_run.values, axis=1)
    etkf_errors = etkf_errors[:,-5000:]
    etkf_errors = etkf_errors[:,np.arange(9,5000,10)]
    B = np.cov(etkf_errors.reshape(system_dim, -1))
    return B

In [7]:
def run_backprop_4dvar(system_dim, nr_steps, spinup_size, valid_size, transient_size, test_size, 
                       test_run, delta_t, obs_location_count, obs_sd, sigma_bg_multiplier, 
                       sigma_obs, analysis_window, analysis_time_in_window, 
                       random_seed, num_iters, learning_rate, lr_decay):
    enkf_run_pkl = './out_rev/l96/l96_enkf_statevec_{}dim_v1.pkl'.format(system_dim)
    np_rng = np.random.default_rng(random_seed)
    jax.clear_caches()

    ### Nature Run
    nature_run = dab.data.Lorenz96(system_dim=system_dim, delta_t=delta_t,
                                   store_as_jax=True, random_seed=random_seed)

    x0_initial = np_rng.normal(size=system_dim, scale=1)
    nature_run.generate(n_steps=nr_steps, x0 = x0_initial) 
    nr_spinup, nr_valid, nr_transient_and_test = nature_run.split_train_valid_test(
        spinup_size, valid_size, transient_size + test_size)
    nr_transient, nr_test, _ = nr_transient_and_test.split_train_valid_test(
        transient_size, test_size, 0)

    if not test_run:
        nr_eval = nr_valid
    else:
        nr_eval = nr_test


    ### Observations
    obs_l96 = dab.observer.Observer(
        nr_eval,
        time_indices = np.arange(0, nr_eval.time_dim, 5),
        random_location_count = obs_location_count,
        error_bias = 0.0,
        error_sd = obs_sd,
        random_seed=random_seed,
        stationary_observers=True,
        store_as_jax=True
    )
    obs_vec_l96 = obs_l96.observe()

    
    ### Forecast Model
    model_l96 = dab.data.Lorenz96(system_dim=system_dim, delta_t=delta_t, 
                                  store_as_jax=True, random_seed=random_seed)

    class L96Model(dab.model.Model):                                                                       
        """Defines model wrapper for Lorenz96 to test forecasting."""
        def forecast(self, state_vec, n_steps):
            self.model_obj.generate(x0=state_vec.values, n_steps=n_steps)
            new_vals = self.model_obj.values 

            new_vec = dab.vector.StateVector(values=new_vals, store_as_jax=True)

            return new_vec

    fc_model = L96Model(model_obj=model_l96)
    
    ### Set up DA matrices: H (observation), R (obs error), B (background error)
    H = np.zeros((obs_location_count, system_dim))
    H[np.arange(H.shape[0]), obs_vec_l96.location_indices[0]] = 1
    R = (sigma_obs**2)* np.identity(obs_location_count)
    B = sigma_bg_multiplier*calc_background_cov(enkf_run_pkl) 

    
    ### Run data assimilation
    da_time_start = timer()
    
    # Prep DA object
    dc = dab.dacycler.Var4DBackprop(
        system_dim=system_dim,
        delta_t=nr_eval.delta_t,
        H=H,
        B=B,
        R=R,
        num_iters=num_iters,
        loss_growth_limit=1000,
        learning_rate=learning_rate,
        lr_decay=lr_decay,
        model_obj=fc_model,
        obs_window_indices=[0,5,10],
        steps_per_window=11, # 11 instead of 10 because inclusive of 0 and 11
        )

    # Generate initial conditions
    cur_tstep = 0
    x0_original = nr_eval.values[cur_tstep] + np_rng.normal(size=(system_dim,), 
                                                            scale=np.diag(B))
    x0_sv = dab.vector.StateVector(
        values=x0_original,
        store_as_jax=True)
    
    # Execute
    out_statevec = dc.cycle(
        input_state = x0_sv,
        start_time = nr_eval.times[cur_tstep],
        obs_vector = obs_vec_l96,
        analysis_window=analysis_window,
        n_cycles=498,
        return_forecast=True,
        obs_error_sd=sigma_obs,
        analysis_time_in_window=analysis_time_in_window)
    
    da_time = timer()-da_time_start
    rmse = np.sqrt(np.mean(np.square(nr_eval.values[:-20] - out_statevec.values)))
    
    return out_statevec, rmse, obs_vec_l96, nr_eval, da_time

### Function: 4DVar

In [8]:
def run_4dvar(system_dim, nr_steps, spinup_size, valid_size, transient_size, test_run, 
              test_size, delta_t, obs_location_count, obs_sd, sigma_bg_multiplier, sigma_obs,
              analysis_window, analysis_time_in_window, random_seed, n_outer_loops):
    enkf_run_pkl = './out_rev/l96/l96_enkf_statevec_{}dim_v1.pkl'.format(system_dim)
    np_rng = np.random.default_rng(random_seed)
    jax.clear_caches()

    ### Nature Run
    nature_run = dab.data.Lorenz96(system_dim=system_dim, delta_t=delta_t,
                                   store_as_jax=True, random_seed=random_seed)

    x0_initial = np_rng.normal(size=system_dim, scale=1)
    nature_run.generate(n_steps=nr_steps, x0 = x0_initial) 
    nr_spinup, nr_valid, nr_transient_and_test = nature_run.split_train_valid_test(
        spinup_size, valid_size, transient_size + test_size)
    nr_transient, nr_test, _ = nr_transient_and_test.split_train_valid_test(
        transient_size, test_size, 0)

    if not test_run:
        nr_eval = nr_valid
    else:
        nr_eval = nr_test


    ### Observations
    obs_l96 = dab.observer.Observer(
        nr_eval,
        time_indices = np.arange(0, nr_eval.time_dim, 5),
        random_location_count = obs_location_count,
        error_bias = 0.0,
        error_sd = obs_sd,
        random_seed=random_seed,
        stationary_observers=True,
        store_as_jax=True
    )
    obs_vec_l96 = obs_l96.observe()

    
    ### Forecast Model
    model_l96 = dab.data.Lorenz96(system_dim=system_dim, delta_t=delta_t, 
                                  store_as_jax=True, random_seed=random_seed)
    class L96Model(dab.model.Model):                                                                       
        """Defines model wrapper for Lorenz96 to test forecasting."""
        def forecast(self, state_vec,n_steps):
            self.model_obj.generate(x0=state_vec.values, n_steps=n_steps)
            new_vals = self.model_obj.values 

            new_vec = dab.vector.StateVector(values=new_vals, store_as_jax=True)

            return new_vec

        def compute_tlm(self, state_vec, n_steps):
            """For 4DVar. Not used for Backprop4DVar"""
            M = self.model_obj.generate(n_steps=n_steps, x0=state_vec.values,
                                        return_tlm=True)
            return M, self.model_obj.values

    fc_model = L96Model(model_obj=model_l96)
    
    ### Set up DA matrices
    
    ### Set up DA matrices: H (observation), R (obs error), B (background error)
    H = np.zeros((obs_location_count, system_dim))
    H[np.arange(H.shape[0]), np.tile(obs_vec_l96.location_indices[0], 1)] = 1
    R = (sigma_obs**2) * np.identity(obs_location_count)
    B = sigma_bg_multiplier*calc_background_cov(enkf_run_pkl) 
    
    ### Run data assimilation
    da_time_start = timer()
    
    # Prep DA object
    dc = dab.dacycler.Var4D(
        system_dim=system_dim,
        delta_t=nature_run.delta_t,
        H=H,
        B=B,
        R=R,
        n_outer_loops=n_outer_loops,
        model_obj=fc_model,
        obs_window_indices=[0,5, 10],
        steps_per_window=11, # 11 instead of 10 because inclusive of 0 and 11
    )
    
    # Generate initial conditions
    cur_tstep = 0
    x0_original = nr_eval.values[cur_tstep] + np_rng.normal(size=(system_dim,), 
                                                            scale=np.diag(B))
    x0_sv = dab.vector.StateVector(
        values=x0_original,
        store_as_jax=True)
    
    # Execute
    out_statevec_4dvar = dc.cycle(
        input_state = x0_sv,
        start_time = nr_eval.times[cur_tstep],
        obs_vector = obs_vec_l96,
        analysis_window=analysis_window,
        n_cycles=498,
        return_forecast=True,
        obs_error_sd=sigma_obs,
        analysis_time_in_window=analysis_time_in_window)
    
    da_time = timer()-da_time_start
    rmse = np.sqrt(np.mean(np.square(nr_eval.values[:-20] - out_statevec_4dvar.values)))
    
    return out_statevec_4dvar, rmse, obs_vec_l96, nr_eval, da_time

### Function: Baserun (no DA)

In [9]:
def run_baserun(system_dim, nr_steps, spinup_size, valid_size, transient_size, test_run,
                test_size, delta_t, random_seed, sigma_bg_multiplier):
    enkf_run_pkl = './out_rev/l96/l96_enkf_statevec_{}dim_v1.pkl'.format(system_dim)
    np_rng = np.random.default_rng(random_seed)
    jax.clear_caches()


    ### Nature Run
    nature_run = dab.data.Lorenz96(system_dim=system_dim, delta_t=delta_t, 
                                   store_as_jax=True, random_seed=random_seed)
    x0_initial = np_rng.normal(size=system_dim, scale=1)
    nature_run.generate(n_steps=nr_steps, x0 = x0_initial)
    
    nr_spinup, nr_valid, nr_transient_and_test = nature_run.split_train_valid_test(
        spinup_size, valid_size, transient_size + test_size)
    nr_transient, nr_test, _ = nr_transient_and_test.split_train_valid_test(
        transient_size, test_size, 0)

    if not test_run:
        nr_eval = nr_valid
    else:
        nr_eval = nr_test
  
    ### Forecast Model
    model_l96 = dab.data.Lorenz96(system_dim=system_dim, delta_t=delta_t,
                                  store_as_jax=True, random_seed=random_seed)


    ### Run
    da_time_start = timer()
    cur_tstep = 0
    B = sigma_bg_multiplier*calc_background_cov(enkf_run_pkl) 
    x0_original = nr_eval.values[cur_tstep] + np_rng.normal(size=(system_dim,), 
                                                            scale=np.diagonal(B))
    
    model_l96.generate(x0=x0_original, n_steps=nr_eval.time_dim)
    
    rmse = np.sqrt(np.mean(np.square(model_l96.values[:-20] - nr_eval.values[:-20])))
    da_time = timer()-da_time_start
    out_vec = dab.vector.StateVector(values=model_l96.values, times=model_l96.times, store_as_jax=True)
    
    return out_vec, rmse, nr_eval, da_time

# Run on Validation

### System size experiments - Validation Set

In [ ]:
# Backprop 4D Var w/ Approximate Hessian
out_dict_list_bp = []
random_seed_list = np.arange(num_runs)
system_dim_list = [6, 20, 36, 72, 144, 256]

for system_dim in system_dim_list:
    
    obs_location_count = int(system_dim/2)
    best_results_filtered = best_results_system_dim.loc[
        best_results_system_dim['system_dim']==system_dim]
    learning_rate = best_results_filtered['config/lr'].values[0]
    lr_decay = best_results_filtered['config/lr_decay'].values[0]

    for i in range(num_runs):
        
        random_seed = system_dim + random_seed_list[i]
        
        run_dict = dict(
            system_dim=system_dim, 
            nr_steps= nr_steps,
            spinup_size=spinup_size,
            valid_size=valid_size,
            transient_size=transient_size,
            test_size=test_size,
            test_run=False,
            delta_t=delta_t,
            obs_location_count=obs_location_count,
            obs_sd=obs_sd,
            sigma_bg_multiplier=5*obs_sd,
            sigma_obs=sigma_obs,
            analysis_window=analysis_window,
            analysis_time_in_window=analysis_time_in_window,
            random_seed=random_seed,
            num_iters=num_iters,
            learning_rate=learning_rate,
            lr_decay = lr_decay) 
        out_bp, error_bp, obs_vec_l96, nr_eval, da_time = run_backprop_4dvar(**run_dict)
        
        run_dict['time'] = da_time 
        run_dict['rmse'] = error_bp
        run_dict['run_num'] = i
        print('Run {}, Time = {}'.format(i,run_dict['time']))
        out_dict_list_bp.append(run_dict)
        
bp_df_time =  pd.DataFrame(out_dict_list_bp)
bp_df_time.to_csv('./out_rev/l96/bp_systemdim_val_{}.csv'.format(output_suffix))

In [11]:
# 4D Var
out_dict_list_4dvar = []

random_seed_list = np.arange(num_runs)
system_dim_list = [6, 20, 36, 72, 144, 256]

for system_dim in system_dim_list:
    
    for i in range(num_runs):
        
        obs_location_count = int(system_dim/2)
        random_seed = system_dim + random_seed_list[i]
        
        run_dict = dict(system_dim=system_dim, 
            nr_steps=nr_steps,
            spinup_size=spinup_size,
            valid_size=valid_size,
            transient_size=transient_size,
            test_size=test_size,
            test_run=False,
            delta_t=delta_t,
            obs_location_count=obs_location_count,
            obs_sd=obs_sd,
            sigma_bg_multiplier=5*obs_sd,
            sigma_obs=sigma_obs,
            analysis_window=analysis_window,
            analysis_time_in_window=analysis_time_in_window,
            random_seed=random_seed,
            n_outer_loops = n_outer_loops)
        
        out_4dvar, error_4dvar, obs_vec_l96, nr_eval, da_time = run_4dvar(**run_dict)
        run_dict['time'] = da_time
        run_dict['rmse'] = error_4dvar
        run_dict['run_num'] = i
        
        print('Run {}, Time = {}'.format(i,run_dict['time']))
        out_dict_list_4dvar.append(run_dict)
        
var4d_df_time =  pd.DataFrame(out_dict_list_4dvar)
var4d_df_time.to_csv('./out_rev/l96/var4d_systemdim_val_{}.csv'.format(output_suffix))

Run 0, Time = 7.439323276000096
Run 0, Time = 8.281533513999989
Run 0, Time = 10.264803503999929
Run 0, Time = 19.419301095000037
Run 0, Time = 56.91310063000003
Run 0, Time = 155.4301503920001


In [12]:

bp_df_time[['system_dim','time','rmse']].groupby('system_dim').mean()

,time,rmse
system_dim,,
6,9.844868,0.411135
20,8.099211,0.459486
36,7.848270,0.397172
72,8.106770,0.820750
144,8.271580,0.950730
256,10.423639,0.972403


In [13]:

var4d_df_time[['system_dim','time','rmse']].groupby('system_dim').mean()

,time,rmse
system_dim,,
6,7.439323,0.664313
20,8.281534,0.564881
36,10.264804,0.462790
72,19.419301,0.793372
144,56.913101,0.921575
256,155.430150,0.945843


### Experiments varying number of observations and obs error - Validation Set

In [14]:
# Backprop 4D Var w/ Approximate Hessian
out_dict_list_bp_obs = []

num_iters=3
system_dim = 36
random_seed = system_dim
num_obs_list = [6, 12, 18, 24, 30, 36]
obs_error_list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0, 1.5, 2.0]

for obs_location_count in num_obs_list:
    for obs_sd in obs_error_list:
        
        best_results_filtered = best_results_heatmap.loc[
            (best_results_heatmap['obs_sd'].round(2)==round(obs_sd, 2)) & 
            (best_results_heatmap['num_obs']==obs_location_count)]
        learning_rate = best_results_filtered['config/lr'].values[0]
        lr_decay = best_results_filtered['config/lr_decay'].values[0]
        
        for i in range(num_runs):
            run_dict = dict(system_dim=system_dim, 
                nr_steps=nr_steps,
                spinup_size=spinup_size,
                valid_size=valid_size,
                transient_size=transient_size,
                test_size=test_size,
                test_run=False,
                delta_t=delta_t,
                obs_location_count=obs_location_count,
                obs_sd=obs_sd,
                sigma_bg_multiplier=5*obs_sd,
                sigma_obs=obs_sd*1.25,
                analysis_window=analysis_window,
                analysis_time_in_window=analysis_time_in_window,
                random_seed=random_seed,
                num_iters=num_iters,
                learning_rate=learning_rate,
                lr_decay=lr_decay)
            
            out_bp, error_bp, obs_vec_l96, nature_run, da_time = run_backprop_4dvar(**run_dict)
            run_dict['time'] = da_time
            run_dict['rmse'] = error_bp
            run_dict['run_num'] = i
            
            print('Run {}, Time = {}'.format(i,run_dict['time']))
            print(obs_location_count, obs_sd, error_bp)
            out_dict_list_bp_obs.append(run_dict)
            
bp_df_obs =  pd.DataFrame(out_dict_list_bp_obs)
bp_df_obs.to_csv('./out_rev/l96/bp_heatmap_val_{}.csv'.format(output_suffix))

Run 0, Time = 8.155514337
6 0.1 3.527367827332716
Run 0, Time = 7.549444963000042
6 0.2 3.526097344362216
Run 0, Time = 8.013145010000017
6 0.3 3.5518560383182227
Run 0, Time = 7.799334356000031
6 0.4 3.6461219324281156
Run 0, Time = 8.12245957999994
6 0.5 3.727740382987345
Run 0, Time = 7.7689600219999875
6 0.75 3.665815636053805
Run 0, Time = 7.805517397000017
6 1.0 3.798091098867156
Run 0, Time = 7.9988638700000365
6 1.5 4.035668717598643
Run 0, Time = 8.002264056000058
6 2.0 4.17021680071887
Run 0, Time = 8.040849258000094
12 0.1 0.6214560966698023
Run 0, Time = 7.686284814000032
12 0.2 1.1735491145682941
Run 0, Time = 8.067776934999983
12 0.3 1.1428448011675825
Run 0, Time = 7.916562366999983
12 0.4 1.6419404068767274
Run 0, Time = 8.065040930999999
12 0.5 1.9623525837898705
Run 0, Time = 7.774395375999916
12 0.75 1.9646932191204685
Run 0, Time = 8.120785591000072
12 1.0 2.284174895988646
Run 0, Time = 7.658006453000098
12 1.5 2.9604521020764034
Run 0, Time = 7.444949548000068
12 

In [15]:
# 4D Var

out_dict_list_4dvar_obs = []

system_dim = 36
random_seed = system_dim
num_obs_list = [6, 12, 18, 24, 30, 36]
obs_error_list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0, 1.5, 2.0]

for obs_location_count in num_obs_list:
    for obs_sd in obs_error_list:
        for i in range(num_runs):
            
            run_dict = dict(system_dim=system_dim, 
                nr_steps=nr_steps,
                spinup_size=spinup_size,
                valid_size=valid_size,
                transient_size=transient_size,
                test_size=test_size,
                test_run=False,
                delta_t=delta_t,
                obs_location_count=obs_location_count,
                obs_sd=obs_sd,
                sigma_bg_multiplier=5*obs_sd,
                sigma_obs=obs_sd*1.25,
                analysis_window=analysis_window,
                analysis_time_in_window=analysis_time_in_window,
                random_seed=random_seed,
                n_outer_loops = n_outer_loops)
            
            out_4dvar, error_4dvar, obs_vec_l96, nature_run, da_time = run_4dvar(**run_dict)
            run_dict['time'] = da_time
            run_dict['rmse'] = error_4dvar
            run_dict['run_num'] = i
            
            print('Run {}, Time = {}'.format(i,run_dict['time']))
            out_dict_list_4dvar_obs.append(run_dict)
            
var4d_df_obs =  pd.DataFrame(out_dict_list_4dvar_obs)
var4d_df_obs.to_csv('./out_rev/l96/var4d_heatmap_val_{}.csv'.format(output_suffix))

Run 0, Time = 10.272369492000053
Run 0, Time = 9.973740922999923
Run 0, Time = 10.337218704999941
Run 0, Time = 10.352962537999929
Run 0, Time = 10.34185001700007
Run 0, Time = 13.518707373000098
Run 0, Time = 11.534863974000018
Run 0, Time = 10.444672313999945
Run 0, Time = 9.769151059000023
Run 0, Time = 10.916559014000086
Run 0, Time = 10.588333597999963
Run 0, Time = 10.370309210999949
Run 0, Time = 10.284058529000049
Run 0, Time = 10.574778471999934
Run 0, Time = 9.988885477000167
Run 0, Time = 10.014318756999955
Run 0, Time = 10.681528258999833
Run 0, Time = 10.185547991000021
Run 0, Time = 11.010830292000037
Run 0, Time = 10.417183695999938
Run 0, Time = 10.130755069000088
Run 0, Time = 10.863737479000065
Run 0, Time = 10.355201555999884
Run 0, Time = 10.278073145000008
Run 0, Time = 10.569839234000028
Run 0, Time = 10.399468636000165
Run 0, Time = 10.503398512999865
Run 0, Time = 10.996139847999984
Run 0, Time = 10.162223186999881
Run 0, Time = 10.676058927999975
Run 0, Time = 

# Repeating everything on the test set    

### System Size Experiments - Test Set

In [10]:
num_runs =  50

In [11]:
# Backprop 4D Var w/ Approximate Hessian

out_dict_list_bp = []

random_seed_list = np.arange(num_runs)+102
system_dim_list = [6, 20, 36, 72, 144, 256]

for system_dim in system_dim_list:
    
    obs_location_count = int(system_dim/2)
    best_results_filtered = best_results_system_dim.loc[best_results_system_dim['system_dim']==system_dim]
    learning_rate = best_results_filtered['config/lr'].values[0]
    lr_decay = best_results_filtered['config/lr_decay'].values[0]

    for i in range(num_runs):
        
        random_seed = system_dim + random_seed_list[i]
        
        run_dict = dict(
            system_dim=system_dim, 
            nr_steps= nr_steps,
            spinup_size=spinup_size,
            valid_size=valid_size,
            transient_size=transient_size,
            test_size=test_size,
            test_run=True,
            delta_t=delta_t,
            obs_location_count=obs_location_count,
            obs_sd=obs_sd,
            sigma_bg_multiplier=5*obs_sd,
            sigma_obs=sigma_obs,
            analysis_window=analysis_window,
            analysis_time_in_window=analysis_time_in_window,
            random_seed=random_seed,
            num_iters=num_iters,
            learning_rate=learning_rate,
            lr_decay = lr_decay)
        
        out_bp, error_bp, obs_vec_l96, nr_eval, da_time = run_backprop_4dvar(**run_dict)
        
        run_dict['time'] = da_time 
        run_dict['rmse'] = error_bp
        run_dict['run_num'] = i
        print('Run {}, Time = {}'.format(i,run_dict['time']))
        out_dict_list_bp.append(run_dict)
        
bp_df_time =  pd.DataFrame(out_dict_list_bp)
bp_df_time.to_csv('./out_rev/l96/bp_systemdim_test_{}.csv'.format(output_suffix))

Run 0, Time = 9.749508412000012
Run 1, Time = 9.965020732000028
Run 2, Time = 10.292765194999959
Run 3, Time = 8.779446807
Run 4, Time = 7.206264593000014
Run 5, Time = 7.207802370999957
Run 6, Time = 7.109483613000009
Run 7, Time = 7.283279720000053
Run 8, Time = 7.3039053069999795
Run 9, Time = 7.201072944000032
Run 10, Time = 7.312599927000065
Run 11, Time = 7.191181712000002
Run 12, Time = 7.338236217000031
Run 13, Time = 7.134680254000045
Run 14, Time = 7.128238790999944
Run 15, Time = 7.183645100000035
Run 16, Time = 7.43621765499995
Run 17, Time = 7.47840209900005
Run 18, Time = 7.452891131000001
Run 19, Time = 7.494407549000016
Run 20, Time = 7.136114699000018
Run 21, Time = 7.327622748999943
Run 22, Time = 7.598913369999991
Run 23, Time = 7.346124463000024
Run 24, Time = 7.278294302999939
Run 25, Time = 7.2713864520000016
Run 26, Time = 7.405388463000008
Run 27, Time = 7.489632447999952
Run 28, Time = 7.071741492000001
Run 29, Time = 7.349974650999911
Run 30, Time = 7.11944426

In [12]:
# 4DVar
out_dict_list_4dvar = []
random_seed_list = np.arange(num_runs)+102
system_dim_list = [6, 20, 36, 72, 144, 256]

for system_dim in system_dim_list:
    for i in range(num_runs):
        obs_location_count = int(system_dim/2)
        random_seed = system_dim + random_seed_list[i]
        run_dict = dict(system_dim=system_dim, 
            nr_steps=nr_steps,
            spinup_size=spinup_size,
            valid_size=valid_size,
            transient_size=transient_size,
            test_size=test_size,
            test_run=True,
            delta_t=delta_t,
            obs_location_count=obs_location_count,
            obs_sd=obs_sd,
            sigma_bg_multiplier=5*obs_sd,
            sigma_obs=sigma_obs,
            analysis_window=analysis_window,
            analysis_time_in_window=analysis_time_in_window,
            random_seed=random_seed,
            n_outer_loops = n_outer_loops)
        out_4dvar, error_4dvar, obs_vec_l96, nr_eval, da_time = run_4dvar(**run_dict)
        run_dict['time'] = da_time
        run_dict['rmse'] = error_4dvar
        run_dict['run_num'] = i
        print('Run {}, Time = {}'.format(i,run_dict['time']))
        out_dict_list_4dvar.append(run_dict)
        
var4d_df_time =  pd.DataFrame(out_dict_list_4dvar)
var4d_df_time.to_csv('./out_rev/l96/var4d_systemdim_test_{}.csv'.format(output_suffix))

Run 0, Time = 7.955367652999939
Run 1, Time = 8.700067325999953
Run 2, Time = 8.01865204700016
Run 3, Time = 7.981755926999995
Run 4, Time = 8.861616913000034
Run 5, Time = 8.02786978999984
Run 6, Time = 7.696603769999911
Run 7, Time = 7.945742597999924
Run 8, Time = 7.669664310999906
Run 9, Time = 7.918389610999839
Run 10, Time = 7.695635843999753
Run 11, Time = 7.947549381999579
Run 12, Time = 7.7652997149998555
Run 13, Time = 7.78310048900039
Run 14, Time = 8.397238012999878
Run 15, Time = 7.812853978000021
Run 16, Time = 7.84215971499998
Run 17, Time = 8.845600783999544
Run 18, Time = 7.866669129000002
Run 19, Time = 7.805695423000543
Run 20, Time = 8.870478607999758
Run 21, Time = 7.762642756999412
Run 22, Time = 7.66334615300002
Run 23, Time = 7.707393423000212
Run 24, Time = 8.475932385000306
Run 25, Time = 7.592664489000526
Run 26, Time = 7.62445083700004
Run 27, Time = 8.794338597000205
Run 28, Time = 8.032549970000218
Run 29, Time = 8.057945943999584
Run 30, Time = 8.76348609

### Experiments varying number of observations and obs error - Test Set

In [13]:
# Backprop 4D Var w/ Approximate Hessian

num_runs =  30
system_dim = 36
num_obs_list = [6, 12, 18, 24, 30, 36]
obs_error_list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0, 1.5, 2.0]

for obs_location_count in num_obs_list:
    for obs_sd in obs_error_list:
        
        best_results_filtered = best_results_heatmap.loc[
            (best_results_heatmap['obs_sd'].round(2)==round(obs_sd, 2)) & 
            (best_results_heatmap['num_obs']==obs_location_count)]
        learning_rate = best_results_filtered['config/lr'].values[0]
        lr_decay = best_results_filtered['config/lr_decay'].values[0]
        out_dict_list_bp_obs = []
        
        for i in range(num_runs):
            
            random_seed = 202 + i
            
            run_dict = dict(system_dim=system_dim, 
                nr_steps=nr_steps,
                spinup_size=spinup_size,
                valid_size=valid_size,
                transient_size=transient_size,
                test_size=test_size,
                test_run=True,
                delta_t=delta_t,
                obs_location_count=obs_location_count,
                obs_sd=obs_sd,
                sigma_bg_multiplier=5*obs_sd,
                sigma_obs=obs_sd*1.25,
                analysis_window=analysis_window,
                analysis_time_in_window=analysis_time_in_window,
                random_seed=random_seed,
                num_iters=num_iters,
                learning_rate=learning_rate,
                lr_decay=lr_decay)
            
            out_bp, error_bp, obs_vec_l96, nature_run, da_time = run_backprop_4dvar(**run_dict)
            
            run_dict['time'] = da_time
            run_dict['rmse'] = error_bp
            run_dict['run_num'] = i
                        
            out_dict_list_bp_obs.append(run_dict)
            
        bp_df_obs = pd.DataFrame(out_dict_list_bp_obs)
        out_csv = './out_rev/l96/bp_heatmap_test_{}.csv'.format(output_suffix)
        bp_df_obs.to_csv(out_csv, mode='a',
                            header=(not os.path.isfile(out_csv)),
                            index=False)

In [14]:
# 4DVar

num_runs = 30
system_dim = 36
num_obs_list = [6, 12, 18, 24, 30, 36]
obs_error_list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0, 1.5, 2.0]


for obs_location_count in num_obs_list:
    for obs_sd in obs_error_list:
        
        out_dict_list_4dvar_obs = []
        
        for i in range(num_runs):
            
            random_seed = 202 + i
            
            run_dict = dict(system_dim=system_dim, 
                nr_steps=nr_steps,
                spinup_size=spinup_size,
                valid_size=valid_size,
                transient_size=transient_size,
                test_size=test_size,
                test_run=True,
                delta_t=delta_t,
                obs_location_count=obs_location_count,
                obs_sd=obs_sd,
                sigma_bg_multiplier=5*obs_sd,
                sigma_obs=obs_sd*1.25,
                analysis_window=analysis_window,
                analysis_time_in_window=analysis_time_in_window,
                random_seed=random_seed,
                n_outer_loops = n_outer_loops)
            
            out_4dvar, error_4dvar, obs_vec_l96, nature_run, da_time = run_4dvar(**run_dict)
            
            run_dict['time'] = da_time
            run_dict['rmse'] = error_4dvar
            run_dict['run_num'] = i
            
            out_dict_list_4dvar_obs.append(run_dict)
            
        var4d_df_obs =  pd.DataFrame(out_dict_list_4dvar_obs)
        out_csv = './out_rev/l96/var4d_heatmap_test_{}.csv'.format(output_suffix)
        var4d_df_obs.to_csv(out_csv, mode='a',
                            header=(not os.path.isfile(out_csv)),
                            index=False)

# Run 36D example and save statevectors for later visualization

In [16]:
# Nature run and no-DA baserun
system_dim = 36
i = 0
obs_location_count = 18
obs_sd = 0.5
random_seed = 202 + i

run_dict = dict(system_dim=system_dim, 
                nr_steps=nr_steps,
                spinup_size=spinup_size,
                valid_size=valid_size,
                transient_size=transient_size,
                test_size=test_size,
                test_run=True,
                delta_t=delta_t,
                random_seed=random_seed,
                sigma_bg_multiplier=5*obs_sd)

out_baserun, error_baserun, nature_run, da_time = run_baserun(**run_dict)

run_dict['time'] = da_time
run_dict['rmse'] = error_baserun
run_dict['run_num'] = i

print('Run {}, Time = {}'.format(i,run_dict['time']))
print(obs_location_count, obs_sd, error_baserun)

# Write statevecs
out_file = './out_rev/l96/l96_baserun_results_18obs_36dim_{}.pkl'.format(output_suffix)
with open(out_file, 'wb') as f: 
     pickle.dump(out_baserun, f) 
f.close()

out_file = './out_rev/l96/l96_baserun_nr_18obs_36dim_{}.pkl'.format(output_suffix)
with open(out_file, 'wb') as f: 
     pickle.dump(nature_run, f) 
f.close()

Run 0, Time = 1.8246347570093349
18 0.5 4.961939524549766


In [17]:
# 4DVar
system_dim = 36
i = 0
obs_location_count = 18
obs_sd = 0.5
random_seed = 202 + i

run_dict = dict(system_dim=system_dim, 
                nr_steps=nr_steps,
                spinup_size=spinup_size,
                valid_size=valid_size,
                transient_size=transient_size,
                test_size=test_size,
                test_run=True,
                delta_t=delta_t,
                obs_location_count=obs_location_count,
                obs_sd=obs_sd,
                sigma_bg_multiplier=5*obs_sd,
                sigma_obs=obs_sd*1.25,
                analysis_window=analysis_window,
                analysis_time_in_window=analysis_time_in_window,
                random_seed=random_seed,
                n_outer_loops = n_outer_loops)

out_4dvar, error_4dvar, obs_vec_l96, nature_run, da_time = run_4dvar(**run_dict)

run_dict['time'] = da_time
run_dict['rmse'] = error_4dvar
run_dict['run_num'] = i

print('Run {}, Time = {}'.format(i,run_dict['time']))
print(obs_location_count, obs_sd, error_4dvar)

# Write statevecs
out_file = './out_rev/l96/l96_4dvar_results_18obs_36dim_{}.pkl'.format(output_suffix)
with open(out_file, 'wb') as f: 
     pickle.dump(out_4dvar, f) 
f.close()

out_file = './out_rev/l96/l96_4dvar_nr_18obs_36dim_{}.pkl'.format(output_suffix)
with open(out_file, 'wb') as f: 
     pickle.dump(nature_run, f) 
f.close()

out_file = './out_rev/l96/l96_4dvar_obsvec_18obs_36dim_{}.pkl'.format(output_suffix)
with open(out_file, 'wb') as f: 
     pickle.dump(obs_vec_l96, f) 
f.close()

Run 0, Time = 15.21421150000242
18 0.5 1.0561442299437243


In [18]:
# Backprop 4DVar w/ approximate Hessian

system_dim = 36
i = 0
obs_location_count = 18
obs_sd = 0.5

best_results_filtered = best_results_heatmap.loc[
    (best_results_heatmap['obs_sd'].round(2)==round(obs_sd, 2)) & 
    (best_results_heatmap['num_obs']==obs_location_count)]
learning_rate = best_results_filtered['config/lr'].values[0]
lr_decay = best_results_filtered['config/lr_decay'].values[0]

random_seed = 202 + i

run_dict = dict(system_dim=system_dim, 
    nr_steps=nr_steps,
    spinup_size=spinup_size,
    valid_size=valid_size,
    transient_size=transient_size,
    test_size=test_size,
    test_run=True,
    delta_t=delta_t,
    obs_location_count=obs_location_count,
    obs_sd=obs_sd,
    sigma_bg_multiplier=5*obs_sd,
    sigma_obs=obs_sd*1.25,
    analysis_window=analysis_window,
    analysis_time_in_window=analysis_time_in_window,
    random_seed=random_seed,
    num_iters=num_iters,
    learning_rate=learning_rate,
    lr_decay=lr_decay)

out_bp, error_bp, obs_vec_l96, nature_run, da_time = run_backprop_4dvar(**run_dict)
run_dict['time'] = da_time
run_dict['rmse'] = error_bp
run_dict['run_num'] = i

print('Run {}, Time = {}'.format(i,run_dict['time']))
print(obs_location_count, obs_sd, error_bp)

# Write statevecs
out_file = './out_rev/l96/l96_bp_results_18obs_36dim_{}.pkl'.format(output_suffix)
with open(out_file, 'wb') as f:  
     pickle.dump(out_bp, f) 
f.close()

Run 0, Time = 13.205944904999342
18 0.5 0.9011262709305115
